# Deliverable 3: reproducción de la versión v6 evaluada

Sube `Deliverable3_ejecucion.zip` en el Paso 2. El controlador y los prompts son los del ZIP original.
Los cambios de esta copia afectan carga de archivos, selección del lote, registro de entorno,
reanudación y conteo de llamadas; no cambian la lógica de agentes ni sus prompts.

Por defecto: **completo v6, 75 preguntas**. Para una demostración corta, ejecuta hasta el Paso 5
y usa las consultas individuales; no ejecutes el lote. La carga de pesos requiere GPU y conexión.
Las dependencias originales no estaban fijadas: esta reproducción registra las versiones usadas,
pero no garantiza salidas idénticas a las históricas. Una nueva corrida requiere nueva evaluación.


# Entregable 3 — Sistema de 4 agentes sobre Qwen3-4B
**IA Generativa (580694) · UdeC · QA sobre normativa de pregrado (RI-FI, RG, Calendario 2026)**

Este notebook corre el **mismo** sistema de agentes que probamos localmente con un modelo simulado, pero ahora con **Qwen3-4B** de verdad. El código de los agentes vive en archivos del repositorio; los *prompts* de cada agente son archivos versionados aparte (carpeta `prompts/`), como exige el enunciado.

**Qué hace, en orden:** (1) revisa la GPU, (2) instala dependencias, (3) sube el código y los datos (el zip), (4) carga Qwen3-4B en 4 bits, (5) activa el recuperador denso E5, (6) prueba rápida mostrando los mensajes **entre** agentes, (7) corre el benchmark (50 congeladas + 25 difíciles) y guarda respuestas y trazas para **corrección manual**.

> El notebook **no** calcula el accuracy solo: guarda las respuestas con una columna `correcto` vacía para que la corrección sea manual, como hemos hecho en todo el proyecto.

## Paso 0 — Revisar la GPU
Usa un entorno con GPU (Entorno de ejecución → Cambiar tipo → T4).

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memoria total (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('⚠️  Sin GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).')

## Paso 1 — Instalar dependencias
Qwen3 necesita `transformers` reciente; `bitsandbytes` permite cargar el modelo en 4 bits (cabe en una T4). No se toca `pandas` (Colab depende de su versión).

In [ ]:
# Solo se actualiza lo necesario; pandas y el resto quedan en la versión que trae Colab.
%pip install -q -U "transformers>=4.51" accelerate bitsandbytes
%pip install -q sentence-transformers
print('Listo. Si Colab lo pide, reinicia el entorno (Runtime → Restart) y vuelve a correr desde el Paso 2.')

## Paso 2 — Subir el código y los datos
Sube `Deliverable3_ejecucion.zip` con el botón de selección. También se admite el ZIP original o un ZIP
del repositorio. Se vuelve a pedir el archivo al ejecutar esta celda; los resultados en Drive se conservan.


In [ ]:
# Subir Deliverable3_ejecucion.zip (incluido en la carpeta Deliverable 3).
# También admite el ZIP original o un ZIP del repositorio completo.
import os, sys, zipfile, tempfile, shutil
from pathlib import Path
from google.colab import files

subidos = files.upload()
zips = [Path('/content') / nombre for nombre in subidos if nombre.lower().endswith('.zip')]
assert len(zips) == 1, 'Sube exactamente un ZIP del trabajo.'
with tempfile.TemporaryDirectory(prefix='d3_alvaro_') as temporal:
    destino = Path(temporal).resolve()
    with zipfile.ZipFile(zips[0]) as archivo:
        for item in archivo.infolist():
            ruta = (destino / item.filename).resolve()
            if not ruta.is_relative_to(destino):
                raise ValueError('El ZIP contiene una ruta fuera de su carpeta.')
        archivo.extractall(destino)
    candidatos = [p.parent.parent for p in destino.rglob('agentes/orquestador.py')
                  if (p.parent.parent / 'prompts/01_enrutador.txt').exists()]
    assert len(candidatos) == 1, f'Se esperaba un sistema; encontrados: {len(candidatos)}'
    BASE = '/content/Deliverable3'
    if Path(BASE).exists():
        shutil.rmtree(BASE)
    shutil.copytree(candidatos[0], BASE, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))

PROMPTS = os.path.join(BASE, 'prompts')
AGENTES = os.path.join(BASE, 'agentes')
if AGENTES not in sys.path:
    sys.path.insert(0, AGENTES)
# Recargar módulos si ya se había usado este notebook en la sesión.
for nombre in ('orquestador', 'tools'):
    sys.modules.pop(nombre, None)
print('Sistema preparado:', BASE)
print(open(os.path.join(AGENTES, 'orquestador.py'), encoding='utf-8').readline().strip())


## Paso 3 — Cargar Qwen3-4B en 4 bits
Cuantización NF4. El cómputo usa float16 en una T4 y bfloat16 cuando la GPU lo permite.
Cada llamada usa temperatura 0,7, top-p 0,8, top-k 20 y min-p 0, con thinking desactivado.
La semilla depende del agente y del prompt. Cambiar el prompt también cambia la semilla.
La igualdad de salidas no está garantizada entre versiones de bibliotecas o hardware distintos.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, hashlib

MODELO = 'Qwen/Qwen3-4B'
# La T4 no tiene soporte nativo de bfloat16: en ese caso se calcula en float16.
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map='auto')
model.eval()
print('Modelo cargado:', MODELO, '| cómputo en', DTYPE)

# Presupuesto de tokens de salida por agente (el Redactor y el Verificador razonan en <pensar>).
MAX_TOKENS = {'enrutador': 256, 'recuperador': 160, 'redactor': 1200, 'verificador': 700}
SEMILLA = 2026   # misma semilla base que el D2

def generar(agente, prompt, max_new_tokens=None):
    """Llamada real a Qwen3-4B. Firma f(agente, prompt)->str que espera el orquestador."""
    msgs = [{'role':'user','content':prompt}]
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    max_new_tokens = max_new_tokens or MAX_TOKENS.get(agente, 512)
    semilla = SEMILLA + int(hashlib.sha256((agente + prompt).encode('utf-8')).hexdigest()[:8], 16) % 1_000_000
    torch.manual_seed(semilla)
    inputs = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, top_p=0.8, top_k=20, min_p=0.0,
                             pad_token_id=tok.eos_token_id)
    gen = out[0][inputs.input_ids.shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

print(generar('prueba', 'Responde solo con la palabra: listo'))

## Paso 4 — Recuperador denso E5
Cargamos `multilingual-e5-small` y precomputamos los vectores de los 201 fragmentos (el campo `texto_embedding` ya viene con el prefijo `passage:`). Reemplazamos el respaldo léxico de la herramienta `buscar` por la búsqueda por coseno, igual que en el Entregable 2.

In [ ]:
from tools import Corpus
from sentence_transformers import SentenceTransformer
import numpy as np

corpus = Corpus(AGENTES)
e5 = SentenceTransformer('intfloat/multilingual-e5-small', device='cuda' if torch.cuda.is_available() else 'cpu')
frag = corpus.frag
P = e5.encode([f['texto_embedding'] for f in frag], normalize_embeddings=True,
              batch_size=64, show_progress_bar=True)

def buscar_e5(consulta, k=5):
    q = e5.encode(['query: ' + str(consulta)], normalize_embeddings=True)[0]
    sims = P @ q
    orden = np.argsort(-sims)
    vistos, salida = set(), []
    for i in orden:
        f = frag[int(i)]
        uid = f.get('unidad_id') or f.get('parent_id')
        if uid in vistos:
            continue
        vistos.add(uid)
        u = corpus.por_id.get(uid, {})
        salida.append({'id': uid, 'texto': u.get('texto', f['texto'])})
        if len(salida) >= k:
            break
    return salida

corpus.buscar = buscar_e5   # inyecta el recuperador denso en la herramienta
print('Recuperador E5 listo. Prueba:')
for h in corpus.buscar('nota minima para aprobar una asignatura', 3):
    print(' ', h['id'], '-', h['texto'][:70])

## Paso 5 — Orquestador + prueba rápida (mensajes entre agentes)
Instanciamos el sistema con el `generar` real y resolvemos **una** pregunta. La traza muestra lo que el video debe enseñar: qué decide el Enrutador, qué acciones y observaciones tiene el Recuperador (ReAct), qué escribe el Redactor y qué decide el Verificador (incluido el chequeo de citas).

In [ ]:
from orquestador import Orquestador
import json

orq = Orquestador(corpus, generar)

def mostrar_traza(r):
    print('TIPO:', r['tipo'], '| ciclos:', r['ciclos'], '| veredicto:', r['veredicto'])
    for p in r['traza']:
        print(f"\n[{p['paso']}] {p['agente'].upper()}" + ('  (fallback)' if p.get('fallback') else ''))
        print('   entrada:', json.dumps(p['entrada'], ensure_ascii=False)[:300])
        sal = p['salida'] if isinstance(p['salida'], str) else json.dumps(p['salida'], ensure_ascii=False)
        print('   salida :', sal[:300])
        if p.get('observacion'):
            print('   obs    :', p['observacion'])
    print('\nRESPUESTA FINAL:\n' + r['respuesta'])

mostrar_traza(orq.responder('¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?'))

Prueba también un caso de **premisa falsa** (debe abstenerse, no inventar). Sirve como caso para el video.

In [ ]:
mostrar_traza(orq.responder('El Artículo 45 del Reglamento Interno de la Facultad de Ingeniería fija la nota mínima de la práctica profesional. ¿Cuál es?'))

## Paso 5c — Elegir variante y conjunto
Por defecto, `ABLACION = None` y `SOLO_DIFICILES = False`: completo v6 sobre 75 preguntas.
Variantes: `sin_fewshot`, `sin_react`, `sin_cot`, `sin_verificador`.
`SOLO_DIFICILES = True` limita a 25 preguntas. Ejecuta esta celda antes de elegir la carpeta de Drive.


In [ ]:
# Configuración de la entrega principal. Cada ablación usa su propia carpeta.
ABLACION = None             # None, 'sin_fewshot', 'sin_react', 'sin_cot', 'sin_verificador'
SOLO_DIFICILES = False      # False: 50 originales + 25 difíciles

from orquestador import ABLACIONES
orq = Orquestador(corpus, generar, ablacion=ABLACION)
print('Variante:', ABLACION or 'completo v6', '| conjuntos:', '25 difíciles' if SOLO_DIFICILES else '75 preguntas')


## Paso 5b — Guardar en Google Drive
Autoriza Drive para guardar en `D3_alvaro_runs`. Cada intento genera una carpeta nueva.
Para reanudar, copia la ruta del intento anterior en `REANUDAR_CARPETA`; el programa rechaza
la reanudación si cambiaron fuentes, variante, conjunto, versiones o revisión del modelo.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import os, json, hashlib, importlib.metadata
drive.mount('/content/drive')

# Para reanudar, pega la ruta exacta impresa por una ejecución anterior.
REANUDAR_CARPETA = None
huellas = {str(p.relative_to(Path(BASE))): hashlib.sha256(p.read_bytes()).hexdigest()
           for carpeta in ('agentes', 'prompts') for p in sorted((Path(BASE)/carpeta).rglob('*'))
           if p.is_file() and p.suffix in ('.py', '.txt', '.json', '.csv')}
configuracion = {'ablacion': ABLACION, 'solo_dificiles': SOLO_DIFICILES,
                 'modelo': MODELO, 'semilla_base': SEMILLA, 'fuentes_sha256': huellas,
                 'versiones': {p: importlib.metadata.version(p) for p in
                    ('torch','transformers','accelerate','bitsandbytes','sentence-transformers')},
                 'revision_modelo': getattr(model.config, '_commit_hash', None)}
carpeta = (Path(REANUDAR_CARPETA) if REANUDAR_CARPETA else
           Path('/content/drive/MyDrive/D3_alvaro_runs') /
           ((ABLACION or 'completo') + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')))
carpeta.mkdir(parents=True, exist_ok=True)
registro = carpeta / 'configuracion_reproduccion.json'
if registro.exists():
    assert json.loads(registro.read_text()) == configuracion, 'La configuración cambió: inicia una corrida nueva.'
else:
    assert not (carpeta/'resultados_agentes_D3.csv').exists(), 'Carpeta histórica sin configuración: no mezclar corridas.'
    registro.write_text(json.dumps(configuracion, ensure_ascii=False, indent=2))
os.chdir(carpeta)
print('Carpeta para guardar/reanudar:', carpeta)


## Paso 6 — Ejecutar el lote seleccionado
Guarda después de cada pregunta. Para reanudar tras perder la sesión, repite la preparación,
elige la misma variante y conjunto y establece `REANUDAR_CARPETA` en el Paso 5b.
La columna `correcto` queda vacía: cada nueva salida requiere evaluación externa.
`llamadas_modelo` excluye eventos automáticos; `eventos_traza` los conserva en un conteo separado.


In [ ]:
import pandas as pd, json, time, os

SALIDA_CSV, SALIDA_TRAZAS = 'resultados_agentes_D3.csv', 'trazas_D3.json'
sets = [('congelado_50', os.path.join(AGENTES, 'test_set_50.csv')),
        ('dificil_25',  os.path.join(AGENTES, 'test_set_dificil_25.csv'))]
if SOLO_DIFICILES:
    sets = [s for s in sets if s[0] == 'dificil_25']
print('Variante:', ABLACION or 'sistema completo (v6)', '| sets:', [s[0] for s in sets])

# Retomar si ya hay avance guardado
filas = pd.read_csv(SALIDA_CSV).to_dict('records') if os.path.exists(SALIDA_CSV) else []
trazas = json.load(open(SALIDA_TRAZAS, encoding='utf-8')) if os.path.exists(SALIDA_TRAZAS) else {}
hechas = {f"{f['set']}:{f['id']}" for f in filas}
print('Ya respondidas:', len(hechas))

def fuentes_de(traza):
    out = set()
    for p in traza:
        obs = p.get('observacion') or {}
        if p['agente'] == 'recuperador':
            if obs.get('traido'): out.add(obs['traido'])
            out.update(obs.get('hits', []))
    return '; '.join(sorted(out))

t_inicio = time.time()
for nombre_set, ruta in sets:
    df = pd.read_csv(ruta)
    print(f'\n=== {nombre_set}: {len(df)} preguntas ===')
    for _, row in df.iterrows():
        qid = f"{nombre_set}:{row['id']}"
        if qid in hechas:
            continue
        t0 = time.time()
        try:
            res = orq.responder(str(row['pregunta']))
            error = ''
        except Exception as e:
            res = {'respuesta': '', 'veredicto': {}, 'ciclos': 0, 'tipo': '', 'traza': list(orq.traza)}
            error = repr(e)
        traza = res['traza']
        filas.append({
            'set': nombre_set, 'ablacion': ABLACION or 'completo', 'id': row['id'], 'categoria': row.get('categoria', ''),
            'pregunta': row['pregunta'], 'gold_dato': row.get('gold_dato', ''), 'gold_fuente': row.get('gold_fuente', ''),
            'respuesta_sistema': res['respuesta'], 'tipo_enrutador': res['tipo'],
            'aprobado_verificador': res['veredicto'].get('aprobado', ''), 'motivo_verificador': res['veredicto'].get('motivo', ''),
            'ciclos': res['ciclos'], 'llamadas_modelo': sum(not p.get('decidido_por') for p in traza), 'eventos_traza': len(traza),
            'fallbacks': sum(1 for p in traza if p.get('fallback')),
            'fuentes_traidas': fuentes_de(traza), 'segundos': round(time.time() - t0, 1),
            'error': error, 'correcto': ''})   # <-- 'correcto' se llena a mano (0/1)
        trazas[qid] = traza
        pd.DataFrame(filas).to_csv(SALIDA_CSV, index=False)
        json.dump(trazas, open(SALIDA_TRAZAS, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
        print(f"  {qid:18s} {res['tipo']:13s} ciclos={res['ciclos']}  {round(time.time()-t0,1)}s {('ERROR '+error) if error else ''}")

res_df = pd.DataFrame(filas)
print(f'\nListo: {len(res_df)} filas en {SALIDA_CSV} ({round((time.time()-t_inicio)/60,1)} min en esta sesión).')

### Resumen del comportamiento (no es el accuracy)
Estas cifras describen **cómo trabajó** el sistema: cuántas veces el Verificador mandó a corregir, cuántas salidas del modelo no se pudieron interpretar (fallbacks) y cuánto tardó. La exactitud real sale de la corrección manual de la columna `correcto`.

In [ ]:
res_df = pd.read_csv(SALIDA_CSV)
print(res_df.groupby('set').agg(preguntas=('id','count'), ciclos_prom=('ciclos','mean'),
      con_correccion=('ciclos', lambda c: int((c > 1).sum())), fallbacks=('fallbacks','sum'),
      seg_prom=('segundos','mean'), errores=('error', lambda e: int(e.fillna('').astype(bool).sum()))).round(2))
print('\nTipos asignados por el Enrutador:')
print(res_df.groupby(['set','tipo_enrutador']).size())

## Paso 7 — Descargar resultados y trazas
`resultados_agentes_D3.csv` (para corregir a mano) y `trazas_D3.json` (mensajes entre agentes, para el repositorio).

In [ ]:
try:
    from google.colab import files
    files.download(SALIDA_CSV)
    files.download(SALIDA_TRAZAS)
except Exception as e:
    print('Descarga manual desde el panel de archivos. Detalle:', e)

## Evaluación y cierre
Conserva cada corrida por separado. Aplica la misma pauta externa a todas las condiciones,
incluidos errores y abstenciones. Las trazas históricas del paquete sirven para inspección,
no sustituyen una nueva ejecución. Consulta `AUDITORIA.md` y `VIDEO.md` en la raíz de Deliverable 3.
